### Setting up the environment with `venv`

It is always recommended to create a new environment for any new project to avoid dependency issues and keep a clean working space. 

Step 1: Create a virtual environment in your project directory:

python3.11 -m venv .venv


Step 2: Activate the virtual environment:  
For Linux/Mac:  
source .venv/bin/activate  


For Windows:  
.venv\Scripts\activate  


Step 3: Install the required libraries using requirements.txt from https://github.itu.dk/Big-Data-Management-2025/python_env

pip install -r requirements.txt

Once your environment is set up and dependencies are installed, you are ready to proceed with your project.

# Assignment 2 - Forecasting the Wind Power Production in Orkney

## Step 0: Imports

In [183]:
# You absolutely need these
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVR
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor

from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

## Step 1: The Data
### Load the data

In [184]:
# Read the data into DataFrames
power_df = pd.read_csv('data/power.csv', parse_dates=["time"], index_col="time")
wind_df = pd.read_csv('data/weather.csv', parse_dates=["time"], index_col="time")

Let's take a look at the resulting dataframes:

In [185]:
power_df.head()

,ANM,Non-ANM,Total
time,,,
2021-12-11 13:39:00+00:00,10.195558,17.102,27.297558
2021-12-11 13:40:00+00:00,10.043559,17.582,27.625559
2021-12-11 13:41:00+00:00,10.961558,17.139,28.100558
2021-12-11 13:42:00+00:00,11.240559,16.194,27.434559
2021-12-11 13:43:00+00:00,10.672559,16.566,27.238559


**NOTE:** <em>This table contains three columns, but closer inspection will reveal a very straigh relationship between those three. Can you spot that?<br>
We are clearly interested in the total power generation, regardless of the source type.</em>
- I am not sure I can spot the relation

In [186]:
wind_df.head()

,Direction,Lead_hours,Source_time,Speed
time,,,,
2021-12-11 15:00:00+00:00,SSE,1,1639227600,11.17600
2021-12-11 18:00:00+00:00,SSW,1,1639238400,8.04672
2021-12-11 21:00:00+00:00,WSW,1,1639249200,11.17600
2021-12-12 00:00:00+00:00,WSW,1,1639260000,8.94080
2021-12-12 03:00:00+00:00,SW,1,1639270800,9.83488


**NOTE:** <em>This table contains four columns, but lead hours and source time are irrelevant here. Can you think why?</em>
- Because the lead time and source time can be afledt from *time*, besides, we are interested in what actual time the wind blows with the given speed, not when the forecast was generated.

**NOTE:** <em>Look at the table's index. Do both data sources contain the same intervals? And if not, what problems could arise when merging the data?</em>
- No - the power generation has an interval of once every minute, and the forecast has an interval of once every three hours. If we merge by time, then there is  large amount of missing data on the forecast-side.

### Merging the data sources

We have two dataframes, one with weather forecast and one with power generation. To do some analysis on the relationship between these two datasets, it might be useful to join (and align) the data.
Joining the two datasets with an inner join means keeping only those records that match their index. Although this will work, you may notice that most of our data is discarded due to the unmatching time intervals. You may want to explore other possible ways to merge the data.

### Identify missing values in 'power table'

In [187]:
""" # Build the full expected index at 1-minute frequency
full_index = pd.date_range(start=power_df.index.min(),
                           end=power_df.index.max(),
                           freq="1T")

# Reindex to full timeline
power_df_full = power_df.reindex(full_index)

# Print amount of NAN values in the full dataframe
print ("Before: ", power_df_full['Total'].isna().sum() ) 

# Interpolate missing values
#power_df_full['Total'] = ( power_df_full['Total'].interpolate(method='time') ) # I chose to not do this, as it would be inaccurate

# Print amount of NAN values in the interpolated full dataframe
print ("After: ", power_df_full['Total'].isna().sum() )  """

' # Build the full expected index at 1-minute frequency\nfull_index = pd.date_range(start=power_df.index.min(),\n                           end=power_df.index.max(),\n                           freq="1T")\n\n# Reindex to full timeline\npower_df_full = power_df.reindex(full_index)\n\n# Print amount of NAN values in the full dataframe\nprint ("Before: ", power_df_full[\'Total\'].isna().sum() ) \n\n# Interpolate missing values\n#power_df_full[\'Total\'] = ( power_df_full[\'Total\'].interpolate(method=\'time\') ) # I chose to not do this, as it would be inaccurate\n\n# Print amount of NAN values in the interpolated full dataframe\nprint ("After: ", power_df_full[\'Total\'].isna().sum() )  '

### Visualize missing power data

In [188]:
""" # Visualize missing data timeline
missing_mask = power_df_full['Total'].isna()

fig, ax = plt.subplots(1,2, figsize=(18,4))

# 1. Timeline of missing values (1 = missing, 0 = present)
ax[0].plot(missing_mask.index, missing_mask.astype(int), drawstyle="steps-post", color="black")
ax[0].set_title("Missing Data Timeline")
ax[0].set_ylabel("Missing (1=True)")
ax[0].set_xlabel("Time")

# 2. Distribution of consecutive gap lengths
gap_lengths = []
current_gap = 0
for val in missing_mask:
    if val:
        current_gap += 1
    elif current_gap > 0:
        gap_lengths.append(current_gap)
        current_gap = 0
if current_gap > 0:
    gap_lengths.append(current_gap)

ax[1].hist(gap_lengths, bins=5, color="tab:red", edgecolor="black")
ax[1].set_title("Distribution of Gap Lengths (minutes)")
ax[1].set_xlabel("Gap length")
ax[1].set_ylabel("Count")

plt.tight_layout()
plt.show()
import matplotlib as mpl
mpl.use("pgf")
plt.savefig("missing_power.pgf")

# Print summary statistics of gap lengths
print("Number of gaps:", len(gap_lengths))
print(pd.Series(gap_lengths).describe())
 """

' # Visualize missing data timeline\nmissing_mask = power_df_full[\'Total\'].isna()\n\nfig, ax = plt.subplots(1,2, figsize=(18,4))\n\n# 1. Timeline of missing values (1 = missing, 0 = present)\nax[0].plot(missing_mask.index, missing_mask.astype(int), drawstyle="steps-post", color="black")\nax[0].set_title("Missing Data Timeline")\nax[0].set_ylabel("Missing (1=True)")\nax[0].set_xlabel("Time")\n\n# 2. Distribution of consecutive gap lengths\ngap_lengths = []\ncurrent_gap = 0\nfor val in missing_mask:\n    if val:\n        current_gap += 1\n    elif current_gap > 0:\n        gap_lengths.append(current_gap)\n        current_gap = 0\nif current_gap > 0:\n    gap_lengths.append(current_gap)\n\nax[1].hist(gap_lengths, bins=5, color="tab:red", edgecolor="black")\nax[1].set_title("Distribution of Gap Lengths (minutes)")\nax[1].set_xlabel("Gap length")\nax[1].set_ylabel("Count")\n\nplt.tight_layout()\nplt.show()\nimport matplotlib as mpl\nmpl.use("pgf")\nplt.savefig("missing_power.pgf")\n\n# Pr

### Manifesting 'time' as a feature

In [189]:
# Convert timeindex to a column 
power_df['Time'] = power_df.index

# Make sure the feature has been added
power_df.info() 

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 109026 entries, 2021-12-11 13:39:00+00:00 to 2022-03-11 13:38:00+00:00
Data columns (total 4 columns):
 #   Column   Non-Null Count   Dtype              
---  ------   --------------   -----              
 0   ANM      109026 non-null  float64            
 1   Non-ANM  109026 non-null  float64            
 2   Total    109026 non-null  float64            
 3   Time     109026 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(3)
memory usage: 4.2 MB


### Joining the two datasets properly:

In [190]:
# Join power and forecast on power

#df = power_df_full.join(wind_df, how="left") # TODO: Missing power values
df = power_df.join(wind_df, how="left")

# Interpolate the forecast speed between each observation, then drop remaining NaN
df['Speed'] = ( df['Speed'].interpolate(method='time') )
df = df.dropna(subset = ['Speed'])
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 108945 entries, 2021-12-11 15:00:00+00:00 to 2022-03-11 13:38:00+00:00
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype              
---  ------       --------------   -----              
 0   ANM          108945 non-null  float64            
 1   Non-ANM      108945 non-null  float64            
 2   Total        108945 non-null  float64            
 3   Time         108945 non-null  datetime64[ns, UTC]
 4   Direction    610 non-null     object             
 5   Lead_hours   610 non-null     float64            
 6   Source_time  610 non-null     float64            
 7   Speed        108945 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(6), object(1)
memory usage: 7.5+ MB


## Step 1.5: Understanding the data / EDA

It can be also useful to plot our datasets to see what relationships they might hold.

In [191]:
""" 
TODO: Remove quotations

# Subplots
fig, ax = plt.subplots(1,3, figsize=(25,4))

# 1. Speed and Power for the last 7 days
# Since we interpolated 'Speed', it is "updated" every minute, and we can pull data with a frequency of once every minute over the last 7 days
ax[0].plot(df["Speed"].tail(int(7*24*60)), label="Speed", color="blue") 
ax[0].plot(df["Total"].tail(int(7*24*60)), label="Power", color="tab:red") 
ax[0].set_title("Windspeed & Power Generation over last 7 days")
ax[0].set_xlabel("Time")
ax[0].tick_params(axis='x', labelrotation = 45)
ax[0].set_ylabel("Windspeed [m/s], Power [MW]")
ax[0].legend()

# 2. Speed vs Total (Power Curve nature)
ax[1].scatter(df["Speed"], df["Total"])
power_curve = df.groupby("Speed").median(numeric_only=True)["Total"]
ax[1].plot(
    power_curve.index, 
    power_curve.values,  
    color="red", 
    linewidth=1, 
    label="Median Power Curve",
    alpha = 0.6
)
ax[1].legend()
ax[1].set_title("Windspeed vs Power")
ax[1].set_ylabel("Power [MW]")
ax[1].set_xlabel("Windspeed [m/s]")

# 3. Speed and Power per Wind Direction
wind_grouped_by_direction = df.groupby("Direction").mean(numeric_only=True).reset_index()
bar_width = 0.5
x = np.arange(len(wind_grouped_by_direction.index))

ax[2].bar(x, wind_grouped_by_direction.Total, width=0.5, label="Power", color="tab:red")
ax[2].bar(x + bar_width, wind_grouped_by_direction.Speed, width=0.5, label="Speed", color="blue")
ax[2].legend()
ax[2].set_xticks(x)
ax[2].set_xticklabels(wind_grouped_by_direction.Direction)
ax[2].tick_params(axis='x', labelrotation = 45)
ax[2].set_title("Speed and Power per Direction");


"""

' \nTODO: Remove quotations\n\n# Subplots\nfig, ax = plt.subplots(1,3, figsize=(25,4))\n\n# 1. Speed and Power for the last 7 days\n# Since we interpolated \'Speed\', it is "updated" every minute, and we can pull data with a frequency of once every minute over the last 7 days\nax[0].plot(df["Speed"].tail(int(7*24*60)), label="Speed", color="blue") \nax[0].plot(df["Total"].tail(int(7*24*60)), label="Power", color="tab:red") \nax[0].set_title("Windspeed & Power Generation over last 7 days")\nax[0].set_xlabel("Time")\nax[0].tick_params(axis=\'x\', labelrotation = 45)\nax[0].set_ylabel("Windspeed [m/s], Power [MW]")\nax[0].legend()\n\n# 2. Speed vs Total (Power Curve nature)\nax[1].scatter(df["Speed"], df["Total"])\npower_curve = df.groupby("Speed").median(numeric_only=True)["Total"]\nax[1].plot(\n    power_curve.index, \n    power_curve.values,  \n    color="red", \n    linewidth=1, \n    label="Median Power Curve",\n    alpha = 0.6\n)\nax[1].legend()\nax[1].set_title("Windspeed vs Power"

**NOTE:** <em>These plots should already give us an intuition of the different relationships between features. It seems clear that there is a positive relationship between the wind speed and the power generation from the turbines, as we obviously suspected. But that relationship is not completely linear. Can you spot that? Finally, it seems like the power generation also depends of where the winds is coming from. Maybe this could also be a useful feature.</em> 

In [192]:
#TODO : Cleanup and remove dependency on 'degrees' column
"""
# Use continious circular values or change to radians to eliminate degrees
fig, ax = plt.subplots(figsize=(10,5))

# ---Hexbin density plot ---
hexbin = ax.hexbin(
    df["Degrees"], df["Total"],
    gridsize=50, cmap="Reds", bins="log"
)

# Colorbar for density
cb = fig.colorbar(hexbin, ax=ax)
cb.set_label("log10(N points)")

# --- Mean Power Curve ---
mean_power_curve = (
    df.groupby("Degrees")["Total"]
      .mean()
      .reset_index()
      .sort_values("Degrees")
)

mean_power_curve = ( mean_power_curve
    .set_index("Degrees")
    .reindex(range(361))
    .interpolate()
)

ax.plot(
    mean_power_curve["Total"], 
    color="black", 
    linewidth=1, 
    label="Mean Power Curve"
)

# Formatting
ax.set_title("Density of Power vs Wind Direction")
ax.set_xlabel("Wind Direction [Degrees]")
ax.set_ylabel("Power [MW]")
ax.set_xlim(0, 360)
ax.set_xticks([0, 45, 90, 135, 180, 225, 270, 315, 360])
ax.set_xticklabels(["N","NE","E","SE","S","SW","W","NW","N"])
ax.legend()


# --- Boxplots (power and speed) --- 
fig2, ax2 = plt.subplots(figsize=(8,5))

# Select the columns you want
box_data = df[["Total", "Speed"]]

# Create the boxplot
ax2.boxplot(
    [box_data["Total"].dropna(), box_data["Speed"].dropna()],
    labels=["Power [MW]", "Speed [m/s]"],
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", color="blue"),
    medianprops=dict(color="red", linewidth=2),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    flierprops=dict(marker="o", markerfacecolor="gray", markersize=4, alpha=0.5)
)

# Formatting
ax2.set_title("Distribution of Power and Wind Speed")
ax2.set_ylabel("Values")

"""

'\n# Use continious circular values or change to radians to eliminate degrees\nfig, ax = plt.subplots(figsize=(10,5))\n\n# ---Hexbin density plot ---\nhexbin = ax.hexbin(\n    df["Degrees"], df["Total"],\n    gridsize=50, cmap="Reds", bins="log"\n)\n\n# Colorbar for density\ncb = fig.colorbar(hexbin, ax=ax)\ncb.set_label("log10(N points)")\n\n# --- Mean Power Curve ---\nmean_power_curve = (\n    df.groupby("Degrees")["Total"]\n      .mean()\n      .reset_index()\n      .sort_values("Degrees")\n)\n\nmean_power_curve = ( mean_power_curve\n    .set_index("Degrees")\n    .reindex(range(361))\n    .interpolate()\n)\n\nax.plot(\n    mean_power_curve["Total"], \n    color="black", \n    linewidth=1, \n    label="Mean Power Curve"\n)\n\n# Formatting\nax.set_title("Density of Power vs Wind Direction")\nax.set_xlabel("Wind Direction [Degrees]")\nax.set_ylabel("Power [MW]")\nax.set_xlim(0, 360)\nax.set_xticks([0, 45, 90, 135, 180, 225, 270, 315, 360])\nax.set_xticklabels(["N","NE","E","SE","S","S

## Step 2: Pipeline and data transformations

Now that we have our data, we need to construct the pipeline to process this data and pass it to our Machine Learning model. For this, you may find useful the Pipeline class from Scikit-Learn.

This class applies a list of transforms to your data, and pass the final state to an estimator (your model). Intermediate steps of the pipeline must be ‘transforms’, that is, they must implement fit and transform methods. The final estimator only needs to implement fit. 

The purpose of the pipeline is to assemble several steps that can be cross-validated together while setting different parameters. For this, it enables setting parameters of the various steps using their names and the parameter name separated by a '__'. 

You can find more information about Scikit-Learn's Pipeline [here](https://scikit-learn.org/stable/modules/compose.html#pipeline).

### Create pipeline conversions for Time and Direction

In [193]:
# Convert Timestamp to numeric 'hour', 'weekday' and 'month' features
def convert_timestamp (df):
        df = df.copy()
        df["Time"] = pd.to_datetime(df["Time"])

        # Convert timestamp to usable integers
        df['Hour'] = df["Time"].dt.hour
        df['Weekday'] = df["Time"].dt.dayofweek
        df['Month'] = df["Time"].dt.month
        return df


# Convert Direction in string to numeric sin and cos
def convert_direction(df): 
    direction_to_degrees = { 'N':   0.0, 'NNE': 22.5, 'NE':  45.0, 'ENE': 67.5, 'E':   90.0, 'ESE': 112.5, 'SE':  135.0, 'SSE': 157.5, 'S':   180.0, 'SSW': 202.5, 'SW':  225.0, 'WSW': 247.5, 'W':   270.0, 'WNW': 292.5, 'NW':  315.0, 'NNW': 337.5 }

    # Create a new column with the mapped degrees
    df['Degrees'] = df['Direction'].map(direction_to_degrees)
    df['Radians'] = df['Direction'].map(direction_to_degrees) * np.pi / 180

    # Define sin and cos columns for continous values of circular direction
    df['Sin'] = np.sin(df['Radians'])
    df['Cos'] = np.cos(df['Radians'])

   # Ensure Time is set as index for new incoming data
    df = df.set_index(pd.to_datetime(df["Time"])) #TODO: change back

    # Interpolate between rows in the same manner as 'Speed'
    df[['Sin', 'Cos']] = ( df[['Sin', 'Cos']] .interpolate(method='time') )
    return df

### Construct pipeline

In [ ]:
# Apply transformations for Timestamp and Direction
time_transformer = FunctionTransformer(convert_timestamp)
direction_transformer = FunctionTransformer(convert_direction)

# Only keep selected features
FEATURES = ["Speed", "Hour", "Weekday", "Month", "Sin", "Cos"]
scaler = ColumnTransformer([ ("scaler", MinMaxScaler(), FEATURES) ]) # Create custom scaler to select only specific (numeric) features

# Construct pipeline
pipeline_base = Pipeline([
    # Transformations
    ("time_features", time_transformer), # Explode timestamp to numeric features
    ("direction_features", direction_transformer), # Transform direction to sin and cos values
    ("scaler", scaler), # Normalize data from 0 to 1
    
    # Estimators
])

### Splitting the data


In [ ]:
# Define raw training features
RAW_FEATURES = ["Speed", "Time", "Direction"]

X = df[RAW_FEATURES] 
y = df["Total"]
dataset = [X, y]

# Split using a timeseries-sensitive method 5 times
tscv = TimeSeriesSplit(n_splits=3)

## Step 3: Tracking your experiments with MLFlow

### Initialize MLFlow 

In [196]:
 # Start MLFlow
mlflow.sklearn.autolog() # This is to help us track scikit learn metrics.
mlflow.set_tracking_uri("http://127.0.0.1:5000") # We set the MLFlow UI to display in our local host.

### Define function for gridsearch prediction

In [ ]:
def gridsearch (model, parameter_grid, name):
    
    # Define pipeline:
    pipeline_grid = Pipeline(
        pipeline_base.steps + [ ("regressor", model) ]
        )
    
    # Define grid search
    grid_search = GridSearchCV (
    estimator = pipeline_grid,
    param_grid = parameter_grid,
    cv = tscv,
    scoring = "neg_mean_absolute_error", 
    n_jobs = 1
    )

    # Initiate grid search and log best run
    mlflow.set_experiment(name)

    # TODO: Change the name! 
    with mlflow.start_run(run_name = (name)): 
        
        grid_search.fit(X, y)
        mlflow.log_metric("Average MAE", -grid_search.best_score_) 
       
        # Log metrics for ALL nested runs 
        results = grid_search.cv_results_
        for i, params in enumerate(results["params"]):
            avg_mae = -results["mean_test_score"][i]  # negate because scoring is neg MAE
            with mlflow.start_run(run_name=f"{name}_set{i}", nested=True):
                mlflow.log_params(params)
                mlflow.log_metric("Average MAE", avg_mae)
    mlflow.end_run()    
    <>

### Linear Regression gridsearch

In [198]:
# Perform gridstyle search on LinearSVR to find hyperparameter

# define model
model = LinearRegression()
name = "LinearRegression_"

# Define paramter grid
param_grid = {
}

#gridsearch(model, param_grid, name)

### LinearSVR gridsearch

In [199]:
# Perform gridstyle search on LinearSVR to find hyperparameter

# define model
model = LinearSVR()
name = "LinearSVR"

# Define paramter grid
param_grid = {
    "regressor__max_iter" : [200, 500, 2000],
    "regressor__C" : [0.1, 1, 10],
    "regressor__epsilon" : [0.1, 0.5, 1]
}

#gridsearch(model, param_grid, name)

### XGBoost gridsearch

In [200]:
# Perform gridstyle search on XGBOost to find hyperparameter

# Define name and model
model = XGBRegressor()
name = "XGBoost"

# Define paramter grid
param_grid = {
    "regressor__n_estimators" : [20, 100, 200],
    "regressor__learning_rate" : [0.05, 0.1, 0.5],
    "regressor__max_depth" : [2, 5, 10],
}

#gridsearch(model, param_grid, name)

2025/12/04 17:38:13 INFO mlflow.sklearn.utils: Logging the 5 best runs, 22 runs will be omitted.


🏃 View run XGBoost_set0 at: http://127.0.0.1:5000/#/experiments/607198407445134402/runs/94e0cd7389b3403ea5ccc982d40b7a06
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/607198407445134402
🏃 View run XGBoost_set1 at: http://127.0.0.1:5000/#/experiments/607198407445134402/runs/00697f91f7ba458f8311b4fb6994af21
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/607198407445134402
🏃 View run XGBoost_set2 at: http://127.0.0.1:5000/#/experiments/607198407445134402/runs/e824370ac1bd49c0b6bbbe860d6f0f33
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/607198407445134402
🏃 View run XGBoost_set3 at: http://127.0.0.1:5000/#/experiments/607198407445134402/runs/4908fce02c9945eb849988c83940c031
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/607198407445134402
🏃 View run XGBoost_set4 at: http://127.0.0.1:5000/#/experiments/607198407445134402/runs/b58c774a9d7043fbbb3bbc0f0a274efa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/607198407445134402
🏃 View run

### Neural Network gridsearch

In [201]:
# Define name and model
model = MLPRegressor()
name = "NeuralNetwork"

# Define parameter grid
param_grid = {
    "regressor__hidden_layer_sizes" : [(64,), (128,)],
    "regressor__max_iter" : [200],
    "regressor__learning_rate_init" : [0.005, 0.01],
}

# Takes way too long with multiple features
#gridsearch(model, param_grid, name)

Chi-square: 1.9396971172550606
p-value: 0.16370142441014984
Degrees of freedom: 1
Expected counts: [[3151051.21721929 3061289.78278071]
 [1291452.78278071 1254664.21721929]]


## Step 4: Using a model on future data

### Define function for training XGBoost
Now that I have found my optimal model and hyperparameters, I may construct a function to train that model specifically

In [ ]:
def train_XGBoost (X, y):
    name = "XGBoost"
    
    # Define grid search
    grid_search = GridSearchCV (
        estimator = Pipeline( 
            pipeline_base.steps + [ ("regressor", XGBRegressor()) ] 
            ),
        param_grid = {"regressor__n_estimators" : [200],
                      "regressor__learning_rate" : [0.05], 
                      "regressor__max_depth" : [2]
            },
        cv = tscv,
        scoring = "neg_mean_absolute_error", 
        n_jobs = 1
    )

    # Initiate grid search and log best run
    mlflow.set_experiment(name)
  
    with mlflow.start_run(run_name = (name )) as run:
        grid_search.fit(X, y)
        best_score = -grid_search.best_score_
        mlflow.log_metric("Average MAE", best_score)
        mlflow.sklearn.log_model(
            sk_model = grid_search.best_estimator_,
            artifact_path="model"
        )
        run_id = run.info.run_id 
    
    return(best_score, grid_search.best_estimator_, run_id)

### Define function to retrain my model on new data

In [203]:
def retrain (new_data):
    df_new = pd.concat( [df, new_data], axis=0)
    
    X_new = df_new[RAW_FEATURES]
    y_new = df_new['Total']
        
    new_score, new_model, run_id = train_XGBoost(X_new, y_new)

    old_model = mlflow.sklearn.load_model("models:/XGBoost/Production")
    old_predictions = old_model.predict(X_new)
    old_score = mean_absolute_error(y_new, old_predictions)

    # Compare metrics
    if new_score < old_score: # Note that a lower MAE is better
        print("**New model is better, promoting now**")
        result = mlflow.register_model(
            f"runs:/{run_id}/model", "XGBoost"
        )

        client = MlflowClient()
        client.transition_model_version_stage(
            name = "XGBoost",
            version = result.version,
            stage = "Production",
            archive_existing_versions = True
        )
    else:
        print("** Old model is better** ")

    # TODO: Can I gridsearch through multiple models and keep the best one?

    # TODO: Use requirements2 or requirements1?
    # TODO: Serve from the cloud

### Create initial model and promote to production

In [ ]:

# Train and log model
best_score, best_model, run_id = train_XGBoost(X, y)

# Register the model in the Model Registry
result = mlflow.register_model(
    f"runs:/{run_id}/model", "XGBoost"
)
print("Registered model version:", result.version)

# Promote this version to Production
client = MlflowClient()
client.transition_model_version_stage(
    name="XGBoost",
    version=result.version,
    stage="Production",
    archive_existing_versions=True
)
print(f"Model version {result.version} promoted to Production")

' from mlflow.tracking import MlflowClient\n\n# Train and log model\nbest_score, best_model, run_id = train_XGBoost(X, y)\n\n# Register the model in the Model Registry\nresult = mlflow.register_model(\n    f"runs:/{run_id}/model", "XGBoost"\n)\nprint("Registered model version:", result.version)\n\n# Promote this version to Production\nclient = MlflowClient()\nclient.transition_model_version_stage(\n    name="XGBoost",\n    version=result.version,\n    stage="Production",\n    archive_existing_versions=True\n)\nprint(f"Model version {result.version} promoted to Production") '

### Retrain initial model and evaluate new one 

In [ ]:

# Read the future dataset. May be replaced by any theoretical set
future_df = pd.read_csv("data/future.csv")

# Parse timestamp
future_df["Time"] = pd.to_datetime(future_df["time"], utc=True)

# Set time as index
future_df = future_df.set_index("time")
# Set time as a feature, too
future_df["Time"] = future_df.index

# Artificially add 'Total' power output, since the model needs that data
future_df["Total"] = df["Total"].mean()

# Remove unnecary columns
future_df = future_df[['Time', 'Direction', 'Speed', "Total"]].copy()

# Retrain
retrain(future_df) 

' # Read the future dataset. May be replaced by any theoretical set\nfuture_df = pd.read_csv("data/future.csv")\n\n# Parse timestamp\nfuture_df["Time"] = pd.to_datetime(future_df["time"], utc=True)\n\n# Set time as index\nfuture_df = future_df.set_index("time")\n# Set time as a feature, too\nfuture_df["Time"] = future_df.index\n\n# Artificially add \'Total\' power output, since the model needs that data\nfuture_df["Total"] = df["Total"].mean()\n\n# Remove unnecary columns\nfuture_df = future_df[[\'Time\', \'Direction\', \'Speed\', "Total"]].copy()\n\n# Retrain\nretrain(future_df) '

## Step 5: Deploying your model

So far, we have trained several models and saved the best for predictions. But we have been working locally. Often times, we will want to deploy our model in such a way that other people can take advantage of it, by sending a request with data and getting back predictions. To do this, we need to create an endopoint for our model to receive data. This is done through deployment of the model.

To deploy our model, we first need to serve the model. Serving a model refers to the process of making a trained machine learning model available to receive input data and provide predictions or inferences based on that data. In other words, when you serve a model, you set it up in a way that it can be queried with new data, and it will produce predictions or outputs based on the patterns it learned during training.

Serving a model is a critical step in the machine learning lifecycle, as it allows you to leverage the model's predictive capabilities in real-world applications. When a model is served, it becomes accessible to applications, websites, or other systems that need to utilize its predictions.

To deploy your model, follow section 3.3 from the Assignment PDF. If you want to learn how to deploy models from the Jupyter Notebook, try out the tutorial notebook at examples/sklearn_elasticnet_wine/train.ipynb. You can find it on the [MLFlow Tutorial](https://mlflow.org/docs/latest/tutorials-and-examples/tutorial.html). 

